AutoEncoder

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Create Synthetic Image Data (100 grayscale images of shape 28x28x1)
X_train = np.random.rand(100, 28, 28, 1).astype(np.float32)

# 2. Build the Encoder Network
encoder_inputs = layers.Input(shape=(28, 28, 1))

x = layers.Conv2D(16, (3, 3), activation='relu', padding='same', strides=2)(encoder_inputs) # -> (14, 14, 16)
x = layers.Conv2D(8, (3, 3), activation='relu', padding='same', strides=2)(x)                # -> (7, 7, 8)

# Compressed Bottleneck Representation (Latent Space)
latent_vector = layers.Conv2D(4, (3, 3), activation='relu', padding='same')(x)              # -> (7, 7, 4)

encoder = models.Model(encoder_inputs, latent_vector, name="Encoder")

# 3. Build the Decoder Network
decoder_inputs = layers.Input(shape=(7, 7, 4))

x = layers.Conv2DTranspose(8, (3, 3), activation='relu', padding='same', strides=2)(decoder_inputs) # -> (14, 14, 8)
x = layers.Conv2DTranspose(16, (3, 3), activation='relu', padding='same', strides=2)(x)              # -> (28, 28, 16)

# Reconstruction Layer (Sigmoid maps output back to [0, 1] pixel values)
decoder_outputs = layers.Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)                 # -> (28, 28, 1)

decoder = models.Model(decoder_inputs, decoder_outputs, name="Decoder")

# 4. Combine Encoder + Decoder into Autoencoder
autoencoder_outputs = decoder(encoder(encoder_inputs))
autoencoder = models.Model(encoder_inputs, autoencoder_outputs, name="Autoencoder")

# 5. Compile and Train
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

# Note: Targets (y) are X_train because Autoencoders reconstruct their own inputs
autoencoder.fit(X_train, X_train, epochs=5, batch_size=16, verbose=1)

# 6. Usage: Compress and Reconstruct
compressed_latent = encoder.predict(X_train[:1], verbose=0)
reconstructed_img = decoder.predict(compressed_latent, verbose=0)

print(f"Original Shape:   {X_train[:1].shape}")
print(f"Bottleneck Shape: {compressed_latent.shape} (Compressed from {28*28*1} to {7*7*4} features)")
print(f"Reconstructed:    {reconstructed_img.shape}")

Model: "Autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Encoder (Functional)            │ (None, 7, 7, 4)        │         1,612 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Decoder (Functional)            │ (None, 28, 28, 1)      │         1,609 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,221 (12.58 KB)

 Trainable params: 3,221 (12.58 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0830
Epoch 2/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0829 
Epoch 3/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0829
Epoch 4/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0828
Epoch 5/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0827
Original Shape:   (1, 28, 28, 1)
Bottleneck Shape: (1, 7, 7, 4) (Compressed from 784 to 196 features)
Reconstructed:    (1, 28, 28, 1)


VAE

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Custom Sampling Layer (Reparameterization Trick)
class Sampling(layers.Layer):
    """Uses (z_mean, z_log_var) to sample z (the latent vector)."""
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.random.normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

# 2. Build the Encoder
latent_dim = 2  # 2D latent space for easy visualization

encoder_inputs = layers.Input(shape=(28, 28, 1))
x = layers.Flatten()(encoder_inputs)
x = layers.Dense(128, activation="relu")(x)

# Two parallel output layers for distribution parameters
z_mean = layers.Dense(latent_dim, name="z_mean")(x)
z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)

# Sample z using the custom layer
z = Sampling()([z_mean, z_log_var])

encoder = models.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")

# 3. Build the Decoder
latent_inputs = layers.Input(shape=(latent_dim,))
x = layers.Dense(128, activation="relu")(latent_inputs)
x = layers.Dense(28 * 28, activation="sigmoid")(x)
decoder_outputs = layers.Reshape((28, 28, 1))(x)

decoder = models.Model(latent_inputs, decoder_outputs, name="decoder")

# 4. Define Custom VAE Model with Loss Step
class VAE(models.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super(VAE, self).__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker = tf.keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = tf.keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = tf.keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [self.total_loss_tracker, self.reconstruction_loss_tracker, self.kl_loss_tracker]

    def train_step(self, data):
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)

            # Reconstruction Loss (Binary Cross-Entropy over pixels)
            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(tf.keras.losses.binary_crossentropy(data, reconstruction), axis=(1, 2))
            )

            # KL Divergence Loss: -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
            kl_loss = -0.5 * tf.reduce_mean(
                tf.reduce_sum(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=1)
            )

            total_loss = reconstruction_loss + kl_loss

        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {m.name: m.result() for m in self.metrics}

# 5. Compile and Train on Dummy Synthetic Data
X_dummy = np.random.rand(100, 28, 28, 1).astype(np.float32)

vae = VAE(encoder, decoder)
vae.compile(optimizer='adam')
vae.fit(X_dummy, epochs=5, batch_size=16, verbose=1)

# 6. Generate NEW Novel Images by Sampling Latent Space
random_latent_point = np.random.normal(size=(1, latent_dim))
generated_image = decoder.predict(random_latent_point, verbose=0)
print(f"Generated new image with shape: {generated_image.shape}")

Epoch 1/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - kl_loss: 1.0398 - reconstruction_loss: 543.8803 - total_loss: 544.9202
Epoch 2/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - kl_loss: 0.3839 - reconstruction_loss: 543.3833 - total_loss: 543.7672
Epoch 3/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - kl_loss: 0.1879 - reconstruction_loss: 543.2239 - total_loss: 543.4118
Epoch 4/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - kl_loss: 0.1442 - reconstruction_loss: 543.1359 - total_loss: 543.2800
Epoch 5/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - kl_loss: 0.1385 - reconstruction_loss: 543.0484 - total_loss: 543.1869
Generated new image with shape: (1, 28, 28, 1)


GAN

In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

latent_dim = 100

# 1. Build the Generator Network
def build_generator():
    model = models.Sequential([
        layers.Input(shape=(latent_dim,)),
        layers.Dense(7 * 7 * 128, activation="relu"),
        layers.Reshape((7, 7, 128)),
        layers.Conv2DTranspose(64, (4, 4), strides=2, padding="same", activation="relu"), # -> (14, 14, 64)
        layers.Conv2DTranspose(1, (4, 4), strides=2, padding="same", activation="tanh")    # -> (28, 28, 1)
    ], name="Generator")
    return model

# 2. Build the Discriminator Network
def build_discriminator():
    model = models.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(64, (3, 3), strides=2, padding="same"),
        layers.LeakyReLU(alpha=0.2),
        layers.Conv2D(128, (3, 3), strides=2, padding="same"),
        layers.LeakyReLU(alpha=0.2),
        layers.Flatten(),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid") # Outputs probability: 1 = Real, 0 = Fake
    ], name="Discriminator")
    return model

# 3. Custom GAN Model with Adversarial Training Loop
class GAN(models.Model):
    def __init__(self, generator, discriminator):
        super(GAN, self).__init__()
        self.generator = generator
        self.discriminator = discriminator
        self.d_loss_tracker = tf.keras.metrics.Mean(name="d_loss")
        self.g_loss_tracker = tf.keras.metrics.Mean(name="g_loss")

    def compile(self, d_optimizer, g_optimizer, loss_fn):
        super(GAN, self).compile()
        self.d_optimizer = d_optimizer
        self.g_optimizer = g_optimizer
        self.loss_fn = loss_fn

    @property
    def metrics(self):
        return [self.d_loss_tracker, self.g_loss_tracker]

    def train_step(self, real_images):
        batch_size = tf.shape(real_images)[0]
        random_latent_vectors = tf.random.normal(shape=(batch_size, latent_dim))

        # --- Train Discriminator ---
        generated_images = self.generator(random_latent_vectors)
        combined_images = tf.concat([generated_images, real_images], axis=0)

        # Labels: 0 for fake images, 1 for real images
        labels = tf.concat([tf.zeros((batch_size, 1)), tf.ones((batch_size, 1))], axis=0)

        with tf.GradientTape() as tape:
            predictions = self.discriminator(combined_images)
            d_loss = self.loss_fn(labels, predictions)

        grads = tape.gradient(d_loss, self.discriminator.trainable_weights)
        self.d_optimizer.apply_gradients(zip(grads, self.discriminator.trainable_weights))

        # --- Train Generator ---
        random_latent_vectors = tf.random.normal(shape=(batch_size, latent_dim))
        # Generator wants Discriminator to think fake images are real (label = 1)
        misleading_labels = tf.ones((batch_size, 1))

        with tf.GradientTape() as tape:
            generated_images = self.generator(random_latent_vectors)
            predictions = self.discriminator(generated_images)
            g_loss = self.loss_fn(misleading_labels, predictions)

        grads = tape.gradient(g_loss, self.generator.trainable_weights)
        self.g_optimizer.apply_gradients(zip(grads, self.generator.trainable_weights))

        self.d_loss_tracker.update_state(d_loss)
        self.g_loss_tracker.update_state(g_loss)
        return {"d_loss": self.d_loss_tracker.result(), "g_loss": self.g_loss_tracker.result()}

# 4. Instantiate and Train
generator = build_generator()
discriminator = build_discriminator()

gan = GAN(generator, discriminator)
gan.compile(
    d_optimizer=tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5),
    g_optimizer=tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5),
    loss_fn=tf.keras.losses.BinaryCrossentropy()
)

# Dummy dataset scaled to [-1, 1] to match Generator's tanh output
X_dummy = (np.random.rand(100, 28, 28, 1) * 2 - 1).astype(np.float32)

gan.fit(X_dummy, epochs=5, batch_size=16, verbose=1)

# 5. Generate New Images
test_noise = tf.random.normal(shape=(1, latent_dim))
generated_sample = generator.predict(test_noise, verbose=0)
print(f"Generated synthetic output shape: {generated_sample.shape}")

Epoch 1/5


/usr/local/lib/python3.13/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - d_loss: 0.6719 - g_loss: 0.6816
Epoch 2/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - d_loss: 0.6228 - g_loss: 0.6513
Epoch 3/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - d_loss: 0.5542 - g_loss: 0.6908
Epoch 4/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - d_loss: 0.4490 - g_loss: 0.7984
Epoch 5/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - d_loss: 0.4636 - g_loss: 0.7121
Generated synthetic output shape: (1, 28, 28, 1)
